In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import squidpy as sq
import warnings
from nichecompass.models import NicheCompass
from nichecompass.utils import extract_gp_dict_from_omics_interaction_databases

# 忽略一些不必要的警告
warnings.filterwarnings("ignore")

# ==============================================================================
# 1. 设置文件路径
# ==============================================================================
# 您的空间转录组数据路径
adata_path = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/st_data/visium_lung/my_spatial_data_annotated.h5ad"

# 您的自定义四元组数据库路径 (即您刚才生成的 my_metabolite_network.csv)
# 确保这个文件里包含您定义的 LDHA-SLC16A1, EGFR 等通路
custom_db_path = "/home/zhangjunyi/xiangmu/nichecompass-main/data/pre_data/siyuanzu/my_metabolite_network.csv"

# 模型和结果保存的文件夹
save_dir = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/st_data/visium_lung/my_model_final_integrated/"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)



In [ ]:
# ==============================================================================
# 2. 加载数据与预处理
# ==============================================================================
print(f"📂 正在加载空间数据: {adata_path}")
adata = sc.read_h5ad(adata_path)

# 确保 counts 层存在 (NicheCompass 训练需要原始计数)
if "counts" not in adata.layers:
    if adata.raw is not None:
        adata.layers["counts"] = adata.raw.X.copy()
    else:
        adata.layers["counts"] = adata.X.copy()

# 构建空间邻接图 (如果尚未构建)
if 'spatial_connectivities' not in adata.obsp:
    print("🌐 构建空间邻接图 (Spatial Neighbors)...")
    try:
        sq.gr.spatial_neighbors(adata, spatial_key="spatial", coord_type="generic", n_neighs=6)
    except:
        sc.pp.neighbors(adata, use_rep='spatial', n_neighbors=6, key_added='spatial')



In [ ]:
# ==============================================================================
# 3. 核心步骤：构建“OmniPath + TMCN自定义”联合数据库
# ==============================================================================
print("\n🔗 正在构建联合配体-受体数据库...")

# --- 3.1 加载默认 OmniPath 数据库 (提供背景知识) ---
print("   -> 正在提取 OmniPath 官方数据库 (背景知识库)...")
try:
    # 自动下载并提取人类(human)的配体-受体互作
    # 这将提供几千对常规通讯，作为模型的"基准"
    omnipath_gp_dict = extract_gp_dict_from_omics_interaction_databases(
        species="human",
        version="v2",
        database="omnipath"
    )
    print(f"      ✅ OmniPath 加载成功，包含 {len(omnipath_gp_dict)} 条背景通路。")
except Exception as e:
    print(f"      ⚠️ OmniPath 加载失败 (可能是网络原因)，将仅使用自定义库。错误: {e}")
    omnipath_gp_dict = {}

# --- 3.2 加载您的自定义数据库 (TMCN 核心通路) ---
print(f"   -> 正在读取您的自定义数据库: {custom_db_path}")
df_custom = pd.read_csv(custom_db_path)

# 自动寻找 Source 和 Target 列
cols = df_custom.columns.tolist()
source_col = next((c for c in cols if "source" in c.lower() and "uniprot" not in c.lower()), None)
target_col = next((c for c in cols if "target" in c.lower() and "uniprot" not in c.lower()), None)

if not source_col or not target_col:
    raise ValueError("❌ 无法在自定义 CSV 中找到 Source/Target 列，请检查文件头。")

# --- 3.3 合并数据库并“加权” (通过特殊命名) ---
print("   -> 正在合并并为自定义通路添加 'TMCN_' 前缀...")
custom_gp_dict = {}

for idx, row in df_custom.iterrows():
    # 转大写以匹配基因名
    l_gene = str(row[source_col]).upper().strip()
    r_gene = str(row[target_col]).upper().strip()
    
    # 【关键策略】：添加 "TMCN_" 前缀
    # 这让您的通路在模型中拥有独立且显著的名字，方便后续单独提取和分析
    # 同时也避免了被 OmniPath 中的同名通路覆盖
    gp_name = f"TMCN_Metabolite_{l_gene}_{r_gene}"
    
    custom_gp_dict[gp_name] = {
        "sources": [l_gene],
        "targets": [r_gene]
    }

# 合并字典 (OmniPath + 自定义)
# 您的自定义通路现在混在其中，但有独特名字
combined_gp_dict = {**omnipath_gp_dict, **custom_gp_dict}

print(f"✅ 数据库合并完成！总通路数: {len(combined_gp_dict)}")
print(f"   其中您的 TMCN 核心通路: {len(custom_gp_dict)} 条")



In [ ]:
# ==============================================================================
# 4. 生成 Mask (将生物学知识映射到数据)
# ==============================================================================
print("\n🛠️ 正在生成基因掩码 (Mask)...")

# 准备基因查找表
var_names_upper = [v.upper() for v in adata.var_names]
var_lookup = {name: i for i, name in enumerate(var_names_upper)}
n_genes = len(adata.var_names)

valid_gp_names = []
gp_source_mask_list = []
gp_target_mask_list = []

# 遍历所有通路
for gp_name, genes in combined_gp_dict.items():
    # 检查基因是否存在于您的数据中
    active_sources = [g for g in genes["sources"] if g in var_lookup]
    active_targets = [g for g in genes["targets"] if g in var_lookup]
    
    # 筛选逻辑：
    # 1. 对于 OmniPath (背景)：要求配体和受体都在数据中，保证质量。
    # 2. 对于 TMCN (您的核心)：只要有一端在就保留，防止核心信号丢失。
    is_custom = gp_name.startswith("TMCN_")
    has_both = (len(active_sources) > 0 and len(active_targets) > 0)
    has_any = (len(active_sources) > 0 or len(active_targets) > 0)
    
    if (is_custom and has_any) or (not is_custom and has_both):
        valid_gp_names.append(gp_name)
        
        # 创建 boolean mask
        s_mask = np.zeros(n_genes, dtype=bool)
        t_mask = np.zeros(n_genes, dtype=bool)
        for g in active_sources: s_mask[var_lookup[g]] = True
        for g in active_targets: t_mask[var_lookup[g]] = True
        
        gp_source_mask_list.append(s_mask)
        gp_target_mask_list.append(t_mask)

# 堆叠成矩阵
if len(valid_gp_names) == 0:
    raise ValueError("❌ 没有找到任何匹配的通路！请检查基因名格式。")

gp_source_mask = np.stack(gp_source_mask_list, axis=1)
gp_target_mask = np.stack(gp_target_mask_list, axis=1)

# 保存到 adata
adata.varm["nichecompass_gp_sources"] = gp_source_mask
adata.varm["nichecompass_gp_targets"] = gp_target_mask
adata.uns["nichecompass_gp_names"] = np.array(valid_gp_names)

print(f"✅ 最终保留有效 GP 数量: {len(valid_gp_names)}")
tmcn_count = sum(1 for name in valid_gp_names if name.startswith('TMCN_'))
print(f"   其中保留的 TMCN 通路: {tmcn_count} 条")



In [ ]:
# ==============================================================================
# 5. 设置模型索引与变量
# ==============================================================================
# 找出所有参与通讯的基因索引
active_source_genes = np.where(gp_source_mask.sum(axis=1) > 0)[0]
active_target_genes = np.where(gp_target_mask.sum(axis=1) > 0)[0]
active_gene_indices = np.unique(np.concatenate([active_source_genes, active_target_genes]))

# 存入 uns 供模型调用
adata.uns["nichecompass_genes_idx"] = active_gene_indices
adata.uns["nichecompass_source_genes_idx"] = active_source_genes
adata.uns["nichecompass_target_genes_idx"] = active_target_genes

# 将这些通讯相关基因标记为 High Variable，强迫模型关注它们
adata.var['highly_variable'] = False
adata.var.iloc[active_gene_indices, adata.var.columns.get_loc('highly_variable')] = True

print("✅ 索引构建完成，准备开始训练。")



In [ ]:
# ==============================================================================
# 6. 初始化并训练模型
# ==============================================================================
# 初始化模型
# 注意：active_gp_names_key 我们给了一个具体名字，防止报错
model = NicheCompass(
    adata,
    spatial_key="spatial",
    key_added="nichecompass",
    gp_names_key="nichecompass_gp_names",
    gp_targets_mask_key="nichecompass_gp_targets",
    gp_sources_mask_key="nichecompass_gp_sources",
    genes_idx_key="nichecompass_genes_idx",
    target_genes_idx_key="nichecompass_target_genes_idx",
    source_genes_idx_key="nichecompass_source_genes_idx",
    active_gp_names_key="nichecompass_active_gp_names", 
    n_addon_gp=10 # 允许模型自己发现10个额外模式
)

print("\n🚀 开始训练模型 (这将花费一些时间，取决于GPU)...")
model.train(
    max_epochs=400,
    batch_size=512,
    accelerator="gpu", 
    devices=1
)



In [ ]:
# ==============================================================================
# 7. 结果提取与保存
# ==============================================================================
print("\n💾 正在保存结果...")

# 1. 提取潜在空间特征 (Latent Representation) -> 这是最重要的输出
# 后续聚类、画图都靠它
adata.obsm["X_nichecompass"] = model.get_latent_representation()

# 2. 保存模型权重
model.save(save_dir, overwrite=True)

# 3. 修复潜在的 None 键问题 (防止 adata 保存报错)
keys_to_fix = [k for k in adata.uns.keys() if k is None]
for k in keys_to_fix:
    val = adata.uns.pop(k)
    adata.uns["nichecompass_unknown_param"] = val

# 4. 保存完整的 adata 对象
adata_save_path = os.path.join(save_dir, "adata.h5ad")
adata.write(adata_save_path)

print("="*50)
print(f"🎉 全部完成！")
print(f"1. 模型权重已保存至: {save_dir}")
print(f"2. 分析结果数据已保存至: {adata_save_path}")
print(f"3. 您的自定义通路 (TMCN_...) 已成功融合并被模型学习。")
print("="*50)

# 后续建议：
# 使用 adata = sc.read_h5ad(adata_save_path) 重新加载
# 查看 adata.obsm["X_nichecompass"] 进行聚类
# 专门提取名字以 "TMCN_" 开头的 GP 进行可视化